In [0]:
%run ../config

In [0]:
%sql
use catalog west_africa_conflict

In [0]:

read_df = (spark.read
            .format('csv')
            .option('header', True)
            .option('inferschema', True)
            .option('mode', 'permissive')
            .option("quote", '"')
            .option("escape", '"')
            .option("multiLine", True)
            .load(raw_file)
            )
print(f"✅ CSV loaded successfully")
print(f"   Total rows: {read_df.count()}")
print(f"   Total columns: {len(read_df.columns)}")
read_df.printSchema()

In [0]:
display(read_df)

In [0]:
print(read_df.columns)

In [0]:


null_counts = read_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in read_df.columns
])
display(null_counts)

In [0]:
df_bronze = read_df \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source", F.lit("UCDP_GED_v25.1"))

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option('overwriteSchema', True)\
    .saveAsTable("west_africa_conflict.bronze.acled_raw")

print(f"✅ Bronze Delta table written to west_africa_conflict.bronze.acled_raw")
print(f"   Total rows: {df_bronze.count()}")

In [0]:
df_check = spark.table(f"{catalog}.{bronze_schema}.acled_raw")
display(df_check.limit(10))